In [ ]:

# ============================================
# CELL 1: ENVIRONMENT SETUP
# ============================================

!pip install -q transformers accelerate scikit-learn pandas numpy matplotlib seaborn

import os
import random
import numpy as np
import pandas as pd
import torch
import transformers
import sklearn

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 50)
print("TRANSITION-AWARE INTENT DRIFT NETWORK")
print("=" * 50)
print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not detected. Enable GPU in Runtime settings.")

print("Random seed:", SEED)
print("Environment setup completed.")


TRANSITION-AWARE INTENT DRIFT NETWORK
PyTorch version: 2.11.0+cu130
Transformers version: 5.20.2
Scikit-learn version: 1.6.1
Device: cuda
GPU: Tesla T4
Random seed: 42
Environment setup completed.


In [ ]:

# ============================================
# CELL 2: DOWNLOAD MULTIWOZ 2.2 (CORRECTED)
# ============================================

import os
import json
import urllib.request

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "budzianowski/multiwoz/master/data/MultiWOZ_2.2"
)

DATA_DIR = "/content/multiwoz22"
os.makedirs(DATA_DIR, exist_ok=True)

# Official MultiWOZ 2.2 split folders and file counts
split_files = {
    "train": ("train", 17),
    "validation": ("dev", 2),
    "test": ("test", 2)
}

for split_name, (folder_name, number_of_files) in split_files.items():
    split_dir = os.path.join(DATA_DIR, folder_name)
    os.makedirs(split_dir, exist_ok=True)

    all_dialogues = []

    for i in range(1, number_of_files + 1):
        filename = f"dialogues_{i:03d}.json"
        path = os.path.join(split_dir, filename)

        if not os.path.exists(path) or os.path.getsize(path) == 0:
            url = f"{BASE_URL}/{folder_name}/{filename}"
            print(f"Downloading {folder_name}/{filename}...")
            urllib.request.urlretrieve(url, path)

        with open(path, "r", encoding="utf-8") as f:
            dialogues = json.load(f)

        all_dialogues.extend(dialogues)

    globals()[f"{split_name}_data"] = all_dialogues

    print(f"{split_name}: {len(all_dialogues)} dialogues")

# Download dialogue act annotations
acts_path = os.path.join(DATA_DIR, "dialog_acts.json")

if not os.path.exists(acts_path) or os.path.getsize(acts_path) == 0:
    acts_url = f"{BASE_URL}/dialog_acts.json"
    print("Downloading dialogue act annotations...")
    urllib.request.urlretrieve(acts_url, acts_path)

with open(acts_path, "r", encoding="utf-8") as f:
    dialogue_acts = json.load(f)

print("Dialogue act annotation files:", len(dialogue_acts))
print("Dataset loading completed successfully.")


train: 8437 dialogues
validation: 1000 dialogues
test: 1000 dialogues
Dialogue act annotation files: 10437
Dataset loading completed successfully.


In [ ]:

# ============================================
# CELL 3: INSPECT DATASET STRUCTURE
# ============================================

# Display one dialogue
sample_dialogue = train_data[0]

print("=" * 60)
print("SAMPLE DIALOGUE")
print("=" * 60)

print("Dialogue ID:", sample_dialogue.get("dialogue_id"))
print("Available keys:", list(sample_dialogue.keys()))

print("\nFirst 3 turns:")
for turn in sample_dialogue.get("turns", [])[:3]:
    print({
        "turn_id": turn.get("turn_id"),
        "speaker": turn.get("speaker"),
        "utterance": turn.get("utterance")
    })

print("\nFirst dialogue annotation:")
sample_id = sample_dialogue.get("dialogue_id")
print("Annotation exists:", sample_id in dialogue_acts)

if sample_id in dialogue_acts:
    annotation = dialogue_acts[sample_id]
    print("Annotation keys:", list(annotation.keys()))

    first_turn_key = next(iter(annotation), None)
    if first_turn_key is not None:
        print("Example turn annotation key:", first_turn_key)
        print("Example annotation:")
        print(json.dumps(
            annotation[first_turn_key],
            indent=2
        )[:2000])

print("\nFirst turn frame structure:")
for turn in sample_dialogue.get("turns", []):
    if turn.get("speaker") == "USER":
        print(json.dumps(turn, indent=2)[:2500])
        break


SAMPLE DIALOGUE
Dialogue ID: PMUL4398.json
Available keys: ['dialogue_id', 'services', 'turns']

First 3 turns:
{'turn_id': '0', 'speaker': 'USER', 'utterance': 'i need a place to dine in the center thats expensive'}
{'turn_id': '1', 'speaker': 'SYSTEM', 'utterance': 'I have several options for you; do you prefer African, Asian, or British food?'}
{'turn_id': '2', 'speaker': 'USER', 'utterance': 'Any sort of food would be fine, as long as it is a bit expensive. Could I get the phone number for your recommendation?'}

First dialogue annotation:
Annotation exists: True
Annotation keys: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11']
Example turn annotation key: 0
Example annotation:
{
  "dialog_act": {
    "Restaurant-Inform": [
      [
        "area",
        "centre"
      ],
      [
        "pricerange",
        "expensive"
      ]
    ]
  },
  "span_info": [
    [
      "Restaurant-Inform",
      "area",
      "centre",
      30,
      36
    ],
    [
      "Restaurant

In [ ]:

# ============================================
# CELL 4: CREATE LABELED TRANSITIONS
# ============================================

from collections import Counter

LABEL2ID = {
    "NO_DRIFT": 0,
    "GOAL_CHANGE": 1,
    "DOMAIN_CHANGE": 2,
    "MULTI_GOAL": 3
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}


def get_current_act_domains(dialogue, turn_id):
    """Get domains appearing in the current user's dialogue acts."""
    dialogue_info = dialogue_acts.get(
        dialogue["dialogue_id"], {}
    )

    turn_act_info = dialogue_info.get(str(turn_id), {})
    acts = turn_act_info.get("dialog_act", {})

    domains = set()

    for act_name in acts:
        parts = act_name.split("-", 1)

        if len(parts) == 2:
            domain = parts[0].lower()

            if domain != "general":
                domains.add(domain)

    return domains


def get_filtered_intents(dialogue, turn_id):
    """Extract active intents for domains expressed in this turn."""
    domains = get_current_act_domains(dialogue, turn_id)
    intents = []

    if not domains:
        return intents, domains

    for turn in dialogue["turns"]:
        if (
            str(turn["turn_id"]) == str(turn_id)
            and turn["speaker"] == "USER"
        ):
            for frame in turn.get("frames", []):
                intent = frame.get("state", {}).get(
                    "active_intent", "NONE"
                )
                service = frame.get("service", "").lower()

                if (
                    service in domains
                    and intent != "NONE"
                    and intent not in intents
                ):
                    intents.append(intent)

            break

    return intents, domains


def final_transition_label(
    previous_intents,
    current_intents,
    previous_domains,
    current_domains
):
    """Apply the project's four-class labeling protocol."""
    prev_intents = set(previous_intents)
    curr_intents = set(current_intents)
    prev_domains = set(previous_domains)
    curr_domains = set(current_domains)

    if not curr_intents:
        return "NO_DRIFT"

    if len(curr_domains) > 1 or len(curr_intents) > 1:
        return "MULTI_GOAL"

    if not prev_intents:
        return "NO_DRIFT"

    if curr_intents.issubset(prev_intents):
        return "NO_DRIFT"

    if prev_domains == curr_domains:
        return "GOAL_CHANGE"

    if not curr_domains.intersection(prev_domains):
        return "DOMAIN_CHANGE"

    return "NO_DRIFT"


def create_transition_dataset(dialogues, split_name):
    rows = []

    for dialogue in dialogues:
        user_turns = [
            turn for turn in dialogue.get("turns", [])
            if turn.get("speaker") == "USER"
        ]

        for previous, current in zip(
            user_turns[:-1], user_turns[1:]
        ):
            prev_intents, prev_domains = get_filtered_intents(
                dialogue, previous["turn_id"]
            )
            curr_intents, curr_domains = get_filtered_intents(
                dialogue, current["turn_id"]
            )

            label = final_transition_label(
                prev_intents,
                curr_intents,
                prev_domains,
                curr_domains
            )

            rows.append({
                "dialogue_id": dialogue["dialogue_id"],
                "previous_turn_id": int(previous["turn_id"]),
                "current_turn_id": int(current["turn_id"]),
                "previous_utterance": previous["utterance"],
                "current_utterance": current["utterance"],
                "previous_intents": prev_intents,
                "current_intents": curr_intents,
                "previous_domains": sorted(prev_domains),
                "current_domains": sorted(curr_domains),
                "drift_label": label,
                "labels": LABEL2ID[label]
            })

    result = pd.DataFrame(rows)
    print(f"\n{split_name}: {len(result):,} transitions")
    print(result["drift_label"].value_counts().to_string())

    return result


train_df = create_transition_dataset(train_data, "TRAIN")
validation_df = create_transition_dataset(
    validation_data, "VALIDATION"
)
test_df = create_transition_dataset(test_data, "TEST")

print("\nTransition datasets created.")
print("Label mapping:", LABEL2ID)



TRAIN: 48,339 transitions
drift_label
NO_DRIFT         35936
DOMAIN_CHANGE     5768
GOAL_CHANGE       5172
MULTI_GOAL        1463

VALIDATION: 6,374 transitions
drift_label
NO_DRIFT         4587
DOMAIN_CHANGE     849
GOAL_CHANGE       749
MULTI_GOAL        189

TEST: 6,372 transitions
drift_label
NO_DRIFT         4604
DOMAIN_CHANGE     802
GOAL_CHANGE       739
MULTI_GOAL        227

Transition datasets created.
Label mapping: {'NO_DRIFT': 0, 'GOAL_CHANGE': 1, 'DOMAIN_CHANGE': 2, 'MULTI_GOAL': 3}


In [ ]:

# ============================================
# CELL 5: TOKENIZE BOTH UTTERANCES
# ============================================

from transformers import AutoTokenizer

BASE_MODEL = "microsoft/deberta-v3-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_transition_dataframe(df, tokenizer, max_length=128):
    previous = tokenizer(
        df["previous_utterance"].astype(str).tolist(),
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )

    current = tokenizer(
        df["current_utterance"].astype(str).tolist(),
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )

    return {
        "previous_input_ids": previous["input_ids"],
        "previous_attention_mask": previous["attention_mask"],
        "current_input_ids": current["input_ids"],
        "current_attention_mask": current["attention_mask"],
        "labels": torch.tensor(
            df["labels"].to_numpy(),
            dtype=torch.long
        )
    }


train_tokens = tokenize_transition_dataframe(
    train_df, tokenizer, MAX_LENGTH
)

validation_tokens = tokenize_transition_dataframe(
    validation_df, tokenizer, MAX_LENGTH
)

test_tokens = tokenize_transition_dataframe(
    test_df, tokenizer, MAX_LENGTH
)

print("Tokenization completed.")
print("Maximum sequence length:", MAX_LENGTH)

for split_name, tokens in [
    ("Train", train_tokens),
    ("Validation", validation_tokens),
    ("Test", test_tokens)
]:
    print(f"\n{split_name}:")
    for key, value in tokens.items():
        print(f"  {key}: {tuple(value.shape)}")

print("\nExample previous utterance:")
print(train_df.iloc[0]["previous_utterance"])

print("\nExample current utterance:")
print(train_df.iloc[0]["current_utterance"])

print("\nExample label:", ID2LABEL[int(train_tokens["labels"][0])])


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

Tokenization completed.
Maximum sequence length: 128

Train:
  previous_input_ids: (48339, 128)
  previous_attention_mask: (48339, 128)
  current_input_ids: (48339, 128)
  current_attention_mask: (48339, 128)
  labels: (48339,)

Validation:
  previous_input_ids: (6374, 128)
  previous_attention_mask: (6374, 128)
  current_input_ids: (6374, 128)
  current_attention_mask: (6374, 128)
  labels: (6374,)

Test:
  previous_input_ids: (6372, 128)
  previous_attention_mask: (6372, 128)
  current_input_ids: (6372, 128)
  current_attention_mask: (6372, 128)
  labels: (6372,)

Example previous utterance:
i need a place to dine in the center thats expensive

Example current utterance:
Any sort of food would be fine, as long as it is a bit expensive. Could I get the phone number for your recommendation?

Example label: NO_DRIFT


In [ ]:

# ============================================
# CELL 6: CUSTOM TA-IDN ARCHITECTURE
# ============================================

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModel

class TransitionAwareIntentDriftNetwork(nn.Module):
    def __init__(
        self,
        model_name=BASE_MODEL,
        num_labels=4,
        dropout=0.2,
        freeze_encoder=False
    ):
        super().__init__()

        # Shared pretrained language encoder
        self.encoder = AutoModel.from_pretrained(model_name)

        hidden_size = self.encoder.config.hidden_size

        # Optional encoder freezing for controlled experiments
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

        # Four-way transition representation:
        # previous, current, absolute difference, interaction
        fusion_size = hidden_size * 4

        self.transition_network = nn.Sequential(
            nn.Linear(fusion_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.classifier = nn.Linear(
            hidden_size // 2,
            num_labels
        )

    def encode_utterance(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Attention-mask-aware mean pooling
        token_embeddings = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(
            token_embeddings.dtype
        )

        summed_embeddings = (token_embeddings * mask).sum(dim=1)
        token_counts = mask.sum(dim=1).clamp(min=1)

        return summed_embeddings / token_counts

    def forward(
        self,
        previous_input_ids,
        previous_attention_mask,
        current_input_ids,
        current_attention_mask
    ):
        h_previous = self.encode_utterance(
            previous_input_ids,
            previous_attention_mask
        )

        h_current = self.encode_utterance(
            current_input_ids,
            current_attention_mask
        )

        difference = torch.abs(h_current - h_previous)
        interaction = h_previous * h_current

        transition_features = torch.cat(
            [
                h_previous,
                h_current,
                difference,
                interaction
            ],
            dim=1
        )

        fused_features = self.transition_network(
            transition_features
        )

        logits = self.classifier(fused_features)

        return logits


model = TransitionAwareIntentDriftNetwork().to(DEVICE)

total_parameters = sum(
    p.numel() for p in model.parameters()
)

trainable_parameters = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("TA-IDN MODEL INITIALIZED")
print("=" * 60)
print("Architecture:", model.__class__.__name__)
print("Encoder:", BASE_MODEL)
print("Hidden size:", model.encoder.config.hidden_size)
print("Number of classes:", len(LABEL2ID))
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Device:", DEVICE)
print("\nTransition features:")
print("  1. Previous-turn representation")
print("  2. Current-turn representation")
print("  3. Absolute representation difference")
print("  4. Element-wise representation interaction")
print("\nModel initialization completed.")


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TA-IDN MODEL INITIALIZED
Architecture: TransitionAwareIntentDriftNetwork
Encoder: microsoft/deberta-v3-base
Hidden size: 768
Number of classes: 4
Total parameters: 186,490,756
Trainable parameters: 186,490,756
Device: cuda

Transition features:
  1. Previous-turn representation
  2. Current-turn representation
  3. Absolute representation difference
  4. Element-wise representation interaction

Model initialization completed.


In [ ]:

# ============================================
# CELL 7: TRAINING CONFIGURATION
# ============================================

from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score

# Training configuration
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
EPOCHS = 3
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

class TransitionDataset(Dataset):
    def __init__(self, tokens):
        self.tokens = tokens

    def __len__(self):
        return len(self.tokens["labels"])

    def __getitem__(self, idx):
        return {
            key: value[idx]
            for key, value in self.tokens.items()
        }


train_dataset = TransitionDataset(train_tokens)
validation_dataset = TransitionDataset(validation_tokens)
test_dataset = TransitionDataset(test_tokens)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# Calculate class weights from training labels only
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.array(sorted(ID2LABEL.keys())),
    y=train_df["labels"].to_numpy()
)

class_weights = torch.tensor(
    class_weights_array,
    dtype=torch.float32,
    device=DEVICE
)

# Optimizer
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Mixed precision on supported GPU
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

print("=" * 55)
print("TA-IDN TRAINING CONFIGURATION")
print("=" * 55)
print("Train examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))
print("Test examples:", len(test_dataset))
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Class weights:")

for class_id, weight in enumerate(class_weights_array):
    print(f"  {ID2LABEL[class_id]}: {weight:.4f}")

print("\nTraining setup completed.")


TA-IDN TRAINING CONFIGURATION
Train examples: 48339
Validation examples: 6374
Test examples: 6372
Batch size: 4
Gradient accumulation: 4
Effective batch size: 16
Epochs: 3
Learning rate: 1e-05
Class weights:
  NO_DRIFT: 0.3363
  GOAL_CHANGE: 2.3366
  DOMAIN_CHANGE: 2.0951
  MULTI_GOAL: 8.2603

Training setup completed.


In [ ]:

import gc
import torch
import torch.nn.functional as F

model.train()
optimizer.zero_grad(set_to_none=True)

batch = next(iter(train_loader))
batch = {key: value.to(DEVICE) for key, value in batch.items()}

# Use BF16 if supported; otherwise use full precision.
use_bf16 = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()

try:
    with torch.amp.autocast(
        device_type=DEVICE.type,
        dtype=torch.bfloat16 if use_bf16 else torch.float32,
        enabled=use_bf16
    ):
        logits = model(
            previous_input_ids=batch["previous_input_ids"],
            previous_attention_mask=batch["previous_attention_mask"],
            current_input_ids=batch["current_input_ids"],
            current_attention_mask=batch["current_attention_mask"]
        )

        loss = F.cross_entropy(
            logits.float(),
            batch["labels"],
            weight=class_weights
        )

    # No GradScaler is needed for BF16 or FP32.
    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), MAX_GRAD_NORM
    )

    finite_logits = torch.isfinite(logits).all().item()
    finite_loss = torch.isfinite(loss).item()
    finite_grad_norm = torch.isfinite(grad_norm).item()

    print("=" * 50)
    print("TA-IDN STABILITY TEST")
    print("=" * 50)
    print("Precision:", "BF16" if use_bf16 else "FP32")
    print("Logits shape:", tuple(logits.shape))
    print("Loss:", round(loss.item(), 6))
    print("Finite logits:", finite_logits)
    print("Finite loss:", finite_loss)
    print("Finite gradient norm:", finite_grad_norm)
    print("Gradient norm:", round(grad_norm.item(), 6))

    if finite_logits and finite_loss and finite_grad_norm:
        print("\nStability test passed.")
    else:
        print("\nNumerical instability detected; do not start training yet.")

except torch.cuda.OutOfMemoryError:
    print("GPU out of memory. We will reduce the batch size.")

finally:
    optimizer.zero_grad(set_to_none=True)

    for name in ["batch", "logits", "loss", "grad_norm"]:
        if name in globals():
            del globals()[name]

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    print("GPU memory allocated (GB):",
          round(torch.cuda.memory_allocated() / 1e9, 3))


TA-IDN STABILITY TEST
Precision: BF16
Logits shape: (4, 4)
Loss: 1.483539
Finite logits: True
Finite loss: True
Finite gradient norm: True
Gradient norm: 39.099091

Stability test passed.
GPU memory allocated (GB): 0.405


In [ ]:

import os
import gc
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

BEST_MODEL_PATH = "/content/ta_idn_best.pt"
history = []
best_val_macro_f1 = -1.0

def run_validation(model, loader):
    model.eval()
    all_predictions = []
    all_labels = []
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }

            with torch.amp.autocast(
                device_type=DEVICE.type,
                dtype=torch.bfloat16,
                enabled=(DEVICE.type == "cuda"
                         and torch.cuda.is_bf16_supported())
            ):
                logits = model(
                    previous_input_ids=batch["previous_input_ids"],
                    previous_attention_mask=batch["previous_attention_mask"],
                    current_input_ids=batch["current_input_ids"],
                    current_attention_mask=batch["current_attention_mask"]
                )

                loss = F.cross_entropy(
                    logits.float(),
                    batch["labels"],
                    weight=class_weights
                )

            total_loss += loss.item()
            all_predictions.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(batch["labels"].cpu().tolist())

    return {
        "loss": total_loss / len(loader),
        "accuracy": accuracy_score(all_labels, all_predictions),
        "macro_f1": f1_score(
            all_labels, all_predictions,
            average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            all_labels, all_predictions,
            average="weighted", zero_division=0
        )
    }

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for step, batch in enumerate(progress):
        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.amp.autocast(
            device_type=DEVICE.type,
            dtype=torch.bfloat16,
            enabled=(DEVICE.type == "cuda"
                     and torch.cuda.is_bf16_supported())
        ):
            logits = model(
                previous_input_ids=batch["previous_input_ids"],
                previous_attention_mask=batch["previous_attention_mask"],
                current_input_ids=batch["current_input_ids"],
                current_attention_mask=batch["current_attention_mask"]
            )

            loss = F.cross_entropy(
                logits.float(),
                batch["labels"],
                weight=class_weights
            )
            loss = loss / GRADIENT_ACCUMULATION_STEPS

        loss.backward()
        running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS

        should_update = (
            (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
            or (step + 1) == len(train_loader)
        )

        if should_update:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), MAX_GRAD_NORM
            )
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        progress.set_postfix(
            loss=f"{running_loss / (step + 1):.4f}"
        )

        del batch, logits, loss

    val_metrics = run_validation(model, validation_loader)

    epoch_result = {
        "epoch": epoch + 1,
        "train_loss": running_loss / len(train_loader),
        **{f"val_{k}": v for k, v in val_metrics.items()}
    }
    history.append(epoch_result)

    print(f"\nEpoch {epoch + 1} results:")
    for key, value in epoch_result.items():
        if key != "epoch":
            print(f"{key}: {value:.4f}")

    if val_metrics["macro_f1"] > best_val_macro_f1:
        best_val_macro_f1 = val_metrics["macro_f1"]
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("Best model saved.")

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print("\nTraining complete.")
print("Best validation macro-F1:", round(best_val_macro_f1, 4))
print("Saved model:", BEST_MODEL_PATH)


Epoch 1/3:   0%|          | 0/12085 [00:00<?, ?it/s]


Epoch 1 results:
train_loss: nan
val_loss: nan
val_accuracy: 0.7196
val_macro_f1: 0.2092
val_weighted_f1: 0.6023
Best model saved.


Epoch 2/3:   0%|          | 0/12085 [00:00<?, ?it/s]

KeyboardInterrupt: 